In [0]:
# Define the location of our uploaded files
volume_path = "/Volumes/workspace/default/online_shop_files"

# List all files in the volume
files = dbutils.fs.ls(volume_path)

# Display file names
for file in files:
    print(file.name)

dim_customer.csv
dim_date.csv
dim_payment.csv
dim_product.csv
dim_status.csv
fact_orders.csv
postgres_incremental_orders.csv


In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

# Location of uploaded CSV files
volume_path = "/Volumes/workspace/default/online_shop_files"

# Read all six CSV files
dim_customer = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/dim_customer.csv")

dim_date = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/dim_date.csv")

dim_payment = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/dim_payment.csv")

dim_product = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/dim_product.csv")

dim_status = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/dim_status.csv")

fact_orders = spark.read.option("header", True).option("inferSchema", True).csv(f"{volume_path}/fact_orders.csv")

print("All six CSV files have been loaded successfully!")

All six CSV files have been loaded successfully!


In [0]:
# Display the structure of the orders table
fact_orders.printSchema()

# Display first five orders
display(fact_orders.limit(5))

# Count records
print("Customers:", dim_customer.count())
print("Dates:", dim_date.count())
print("Products:", dim_product.count())
print("Payments:", dim_payment.count())
print("Statuses:", dim_status.count())
print("Orders:", fact_orders.count())

root
 |-- order_id: integer (nullable = true)
 |-- date_key: integer (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- payment_key: integer (nullable = true)
 |-- status_key: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- source: string (nullable = true)



order_id,date_key,customer_key,product_key,payment_key,status_key,quantity,amount,source
100001,20260901,268,3,1,2,1,77635.24,web
100002,20250228,102,5,2,2,1,5562.63,web
100003,20260708,802,2,2,5,1,2055.15,web
100004,20250222,1549,1,4,2,1,7414.43,web
100005,20260112,165,2,4,2,4,6784.53,web


Customers: 2057
Dates: 730
Products: 6
Payments: 4
Statuses: 5
Orders: 199989


In [0]:
# Define the six tables
tables = {
    "dim_customer": dim_customer,
    "dim_date": dim_date,
    "dim_product": dim_product,
    "dim_payment": dim_payment,
    "dim_status": dim_status,
    "fact_orders": fact_orders
}

# Save each DataFrame as a Delta table
for table_name, dataframe in tables.items():

    dataframe.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"workspace.default.{table_name}")

    print(f"Created Delta table: {table_name}")

Created Delta table: dim_customer
Created Delta table: dim_date
Created Delta table: dim_product
Created Delta table: dim_payment
Created Delta table: dim_status
Created Delta table: fact_orders


In [0]:
# List all tables in the default schema
spark.sql("SHOW TABLES IN workspace.default").show(truncate=False)

+--------+------------------+-----------+
|database|tableName         |isTemporary|
+--------+------------------+-----------+
|default |dim_customer      |false      |
|default |dim_date          |false      |
|default |dim_payment       |false      |
|default |dim_product       |false      |
|default |dim_status        |false      |
|default |fact_orders       |false      |
|default |fact_orders_demo  |false      |
|        |incremental_orders|true       |
+--------+------------------+-----------+



In [0]:
for table_name in tables.keys():

    count = spark.table(
        f"workspace.default.{table_name}"
    ).count()

    print(f"{table_name}: {count} records")

dim_customer: 2057 records
dim_date: 730 records
dim_product: 6 records
dim_payment: 4 records
dim_status: 5 records
fact_orders: 199989 records


In [0]:
# View the history of the fact_orders Delta table
history = spark.sql("""
    DESCRIBE HISTORY workspace.default.fact_orders
""")

display(history)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-11-06T13:56:40.000Z,74421546532103,khattakhooria68@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1287768686759789),3df4dc8e-8715-4d9b-8286-e1b8363c16e2,1106-135136-iak4fooa-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 1746494, numDeletionVectorsRemoved -> 1, numOutputRows -> 199989, numOutputBytes -> 1654233)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T13:53:44.000Z,74421546532103,khattakhooria68@gmail.com,MERGE,"Map(predicate -> [""(order_id#13999 = order_id#13606)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1287768686759789),f84b9ca3-faf2-42f8-9a74-0e344b3bdeed,1106-135136-iak4fooa-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 92261, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 8593, executionTimeMs -> 3796, materializeSourceTimeMs -> 971, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 989, numTargetRowsUpdated -> 8593, numOutputRows -> 8593, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8593, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1754)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T13:52:29.000Z,74421546532103,khattakhooria68@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1287768686759789),b405ffe1-6b70-4c0a-adc3-52e88bbfa209,1106-135136-iak4fooa-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1654233, numDeletionVectorsRemoved -> 0, numOutputRows -> 199989, numOutputBytes -> 1654233)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T13:25:15.000Z,74421546532103,khattakhooria68@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1287768686759789),28aa17c8-a3a1-4870-95c5-c6b5249a651c,1105-132016-pw0b8o0v-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 199989, numOutputBytes -> 1654233)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
# Remove an old demo table if you previously created one
spark.sql("""
    DROP TABLE IF EXISTS workspace.default.fact_orders_demo
""")

# Create a separate Delta table
spark.sql("""
    CREATE TABLE workspace.default.fact_orders_demo
    USING DELTA
    AS SELECT * FROM workspace.default.fact_orders
""")

print("Demo table created!")

Demo table created!


In [0]:
# Get one order ID
demo_order_id = (
    spark.table("workspace.default.fact_orders_demo")
    .select("order_id")
    .orderBy("order_id")
    .first()["order_id"]
)

# Update its amount
spark.sql(f"""
    UPDATE workspace.default.fact_orders_demo
    SET amount = amount + 1.0
    WHERE order_id = {demo_order_id}
""")

print("Demo order updated!")

Demo order updated!


In [0]:
# time Travel

# Original version before the update
old_version = spark.sql(f"""
    SELECT order_id, amount
    FROM workspace.default.fact_orders_demo
    VERSION AS OF 0
    WHERE order_id = {demo_order_id}
""")

# Latest version after the update
new_version = spark.sql(f"""
    SELECT order_id, amount
    FROM workspace.default.fact_orders_demo
    WHERE order_id = {demo_order_id}
""")

print("Original version:")
display(old_version)

print("Updated version:")
display(new_version)

Original version:


order_id,amount
100001,77635.24


Updated version:


order_id,amount
100001,77636.24


In [0]:
# Path to the incremental orders file
incremental_path = (
    "/Volumes/workspace/default/online_shop_files/"
    "postgres_incremental_orders.csv"
)

# Read the incremental CSV file
incremental_orders = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(incremental_path)
)

# Display its structure and sample records
incremental_orders.printSchema()
display(incremental_orders.limit(5))

# Count incremental records
print("Incremental records:", incremental_orders.count())


#incremental file contains orders that were updated on or after 1 October 2026

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
166585,567,2026-09-24,Home,3,80021.99,delivered,card,2026-10-01T00:04:36.000Z
188311,1231,2026-09-28,Electronics,2,32242.92,delivered,wallet,2026-10-01T00:06:01.000Z
218039,245,2026-09-25,Books,1,1057.99,delivered,cash_on_delivery,2026-10-01T00:12:26.000Z
261988,1626,2026-09-09,Beauty,5,9547.4,returned,card,2026-10-01T00:13:40.000Z
168034,956,2026-09-30,Books,2,6786.9,pending,wallet,2026-10-01T00:13:49.000Z


Incremental records: 8593


In [0]:
# Display the columns and data types
incremental_orders.printSchema()

# Check the first five records
display(incremental_orders.limit(5))

# Check the date range
incremental_orders.selectExpr(
    "MIN(updated_at) AS earliest_update",
    "MAX(updated_at) AS latest_update"
).show()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- updated_at: timestamp (nullable = true)



order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
166585,567,2026-09-24,Home,3,80021.99,delivered,card,2026-10-01T00:04:36.000Z
188311,1231,2026-09-28,Electronics,2,32242.92,delivered,wallet,2026-10-01T00:06:01.000Z
218039,245,2026-09-25,Books,1,1057.99,delivered,cash_on_delivery,2026-10-01T00:12:26.000Z
261988,1626,2026-09-09,Beauty,5,9547.4,returned,card,2026-10-01T00:13:40.000Z
168034,956,2026-09-30,Books,2,6786.9,pending,wallet,2026-10-01T00:13:49.000Z


+-------------------+-------------------+
|    earliest_update|      latest_update|
+-------------------+-------------------+
|2026-10-01 00:04:36|2026-10-24 23:59:00|
+-------------------+-------------------+



In [0]:
# Check how many incremental order IDs already exist
# in the current fact_orders table

existing_orders = spark.table("workspace.default.fact_orders")

matching_orders = (
    incremental_orders
    .select("order_id")
    .join(
        existing_orders.select("order_id"),
        on="order_id",
        how="inner"
    )
)

print("Incremental records:", incremental_orders.count())
print("Orders already in fact_orders:", matching_orders.count())

print(
    "Distinct incremental order IDs:",
    incremental_orders.select("order_id").distinct().count()
)

Incremental records: 8593
Orders already in fact_orders: 8593
Distinct incremental order IDs: 8593


In [0]:
from pyspark.sql.functions import col, to_date, date_format

# Convert order_date into a proper date
incremental_clean = (
    incremental_orders
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn(
        "date_key",
        date_format(col("order_date"), "yyyyMMdd").cast("int")
    )
)

# Join customer dimension using SCD Type 2 date ranges
inc_customer = (
    incremental_clean.alias("i")
    .join(
        dim_customer.alias("c"),
        (col("i.customer_id") == col("c.customer_id")) &
        (col("i.order_date") >= col("c.effective_from").cast("date")) &
        (col("i.order_date") <= col("c.effective_to").cast("date")),
        "left"
    )
    .select("i.*", col("c.customer_key"))
)

# Join product dimension
inc_product = (
    inc_customer.alias("i")
    .join(
        dim_product.alias("p"),
        col("i.category") == col("p.category"),
        "left"
    )
    .select("i.*", col("p.product_key"))
)

# Join payment dimension
inc_payment = (
    inc_product.alias("i")
    .join(
        dim_payment.alias("p"),
        col("i.payment_method") == col("p.payment_method"),
        "left"
    )
    .select("i.*", col("p.payment_key"))
)

# Join status dimension
inc_status = (
    inc_payment.alias("i")
    .join(
        dim_status.alias("s"),
        col("i.status") == col("s.status"),
        "left"
    )
    .select("i.*", col("s.status_key"))
)

# Select the fact table columns
incremental_fact = inc_status.select(
    "order_id",
    "date_key",
    "customer_key",
    "product_key",
    "payment_key",
    "status_key",
    "quantity",
    "amount"
)

print("Prepared incremental fact records:", incremental_fact.count())

# Check for missing dimension keys
incremental_fact.select([
    col(c).isNull().cast("int").alias(c)
    for c in [
        "date_key", "customer_key", "product_key",
        "payment_key", "status_key"
    ]
]).groupBy().sum().show()

Prepared incremental fact records: 8593
+-------------+-----------------+----------------+----------------+---------------+
|sum(date_key)|sum(customer_key)|sum(product_key)|sum(payment_key)|sum(status_key)|
+-------------+-----------------+----------------+----------------+---------------+
|            0|                0|               0|               0|              0|
+-------------+-----------------+----------------+----------------+---------------+



In [0]:
print("Incremental records:", incremental_fact.count())

print(
    "Distinct order IDs:",
    incremental_fact.select("order_id").distinct().count()
)

print(
    "Records with missing dimension keys:",
    incremental_fact.filter(
        col("customer_key").isNull() |
        col("product_key").isNull() |
        col("payment_key").isNull() |
        col("status_key").isNull() |
        col("date_key").isNull()
    ).count()
)

Incremental records: 8593
Distinct order IDs: 8593
Records with missing dimension keys: 0


In [0]:
from delta.tables import DeltaTable

# Reference the existing Delta table
target_table = DeltaTable.forName(
    spark,
    "workspace.default.fact_orders"
)

# Register incremental records as a temporary view
incremental_fact.createOrReplaceTempView("incremental_orders")

# Update existing records using order_id
spark.sql("""
    MERGE INTO workspace.default.fact_orders AS target
    USING incremental_orders AS source
    ON target.order_id = source.order_id

    WHEN MATCHED THEN UPDATE SET
        target.date_key = source.date_key,
        target.customer_key = source.customer_key,
        target.product_key = source.product_key,
        target.payment_key = source.payment_key,
        target.status_key = source.status_key,
        target.quantity = source.quantity,
        target.amount = source.amount
""")

print("Incremental MERGE completed successfully!")

Incremental MERGE completed successfully!


In [0]:
updated_fact = spark.table("workspace.default.fact_orders")

print("Total orders:", updated_fact.count())

print(
    "Distinct order IDs:",
    updated_fact.select("order_id").distinct().count()
)

print(
    "Duplicate order IDs:",
    updated_fact.count() -
    updated_fact.select("order_id").distinct().count()
)

Total orders: 199989
Distinct order IDs: 199989
Duplicate order IDs: 0


In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY workspace.default.fact_orders
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-06T13:57:07.000Z,74421546532103,khattakhooria68@gmail.com,MERGE,"Map(predicate -> [""(order_id#18700 = order_id#18374)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1287768686759789),f6452404-38c6-40fc-8178-812b3ccdb590,1106-135136-iak4fooa-v2n,3,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 92261, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 8593, executionTimeMs -> 3029, materializeSourceTimeMs -> 702, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 799, numTargetRowsUpdated -> 8593, numOutputRows -> 8593, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8593, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1495)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-06T13:56:40.000Z,74421546532103,khattakhooria68@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1287768686759789),3df4dc8e-8715-4d9b-8286-e1b8363c16e2,1106-135136-iak4fooa-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 1746494, numDeletionVectorsRemoved -> 1, numOutputRows -> 199989, numOutputBytes -> 1654233)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T13:53:44.000Z,74421546532103,khattakhooria68@gmail.com,MERGE,"Map(predicate -> [""(order_id#13999 = order_id#13606)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(1287768686759789),f84b9ca3-faf2-42f8-9a74-0e344b3bdeed,1106-135136-iak4fooa-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 92261, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 8593, executionTimeMs -> 3796, materializeSourceTimeMs -> 971, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 989, numTargetRowsUpdated -> 8593, numOutputRows -> 8593, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 8593, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1754)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-06T13:52:29.000Z,74421546532103,khattakhooria68@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1287768686759789),b405ffe1-6b70-4c0a-adc3-52e88bbfa209,1106-135136-iak4fooa-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1654233, numDeletionVectorsRemoved -> 0, numOutputRows -> 199989, numOutputBytes -> 1654233)",null,Databricks-Runtime/19.13.x-aarch64-

In [0]:
# Read the fact table before incremental loading
fact_before = spark.sql("""
    SELECT *
    FROM workspace.default.fact_orders
    VERSION AS OF 1
""")

# Read the fact table after incremental loading
fact_after = spark.sql("""
    SELECT *
    FROM workspace.default.fact_orders
    VERSION AS OF 2
""")

print("Orders before incremental loading:", fact_before.count())
print("Orders after incremental loading:", fact_after.count())

Orders before incremental loading: 199989
Orders after incremental loading: 199989


In [0]:
# Find an order that was updated
updated_order_id = (
    incremental_fact
    .select("order_id")
    .first()["order_id"]
)

print("Order ID:", updated_order_id)

print("Before incremental loading:")
display(
    spark.sql(f"""
        SELECT *
        FROM workspace.default.fact_orders
        VERSION AS OF 1
        WHERE order_id = {updated_order_id}
    """)
)

print("After incremental loading:")
display(
    spark.sql(f"""
        SELECT *
        FROM workspace.default.fact_orders
        VERSION AS OF 2
        WHERE order_id = {updated_order_id}
    """)
)

Order ID: 166585
Before incremental loading:


order_id,date_key,customer_key,product_key,payment_key,status_key,quantity,amount,source
166585,20260924,583,6,2,2,3,80021.99,web


After incremental loading:


order_id,date_key,customer_key,product_key,payment_key,status_key,quantity,amount,source
166585,20260924,583,6,2,2,3,80021.99,web


In [0]:
# Compare the old and new fact table values
comparison = (
    fact_before.alias("old")
    .join(
        fact_after.alias("new"),
        on="order_id",
        how="inner"
    )
)

# Count records where at least one value changed
from pyspark.sql.functions import (
    col, sum as spark_sum, when
)

changed_records = comparison.filter(
    (col("old.amount") != col("new.amount")) |
    (col("old.quantity") != col("new.quantity")) |
    (col("old.status_key") != col("new.status_key")) |
    (col("old.payment_key") != col("new.payment_key")) |
    (col("old.product_key") != col("new.product_key")) |
    (col("old.customer_key") != col("new.customer_key"))
)

print("Orders with changed values:", changed_records.count())

Orders with changed values: 0


In [0]:
# Create a separate demo table from the current fact table
spark.sql("""
    DROP TABLE IF EXISTS workspace.default.fact_orders_update_demo
""")

spark.sql("""
    CREATE TABLE workspace.default.fact_orders_update_demo
    USING DELTA
    AS SELECT * FROM workspace.default.fact_orders
""")

print("Demo table created!")

Demo table created!


In [0]:
# Select one order
demo_id = (
    spark.table("workspace.default.fact_orders_update_demo")
    .select("order_id")
    .first()["order_id"]
)

# Save the original amount
original_amount = (
    spark.table("workspace.default.fact_orders_update_demo")
    .filter(col("order_id") == demo_id)
    .first()["amount"]
)

print("Order ID:", demo_id)
print("Original amount:", original_amount)

Order ID: 100001
Original amount: 77635.24


In [0]:
# Increase amount by 100 for demonstration
spark.sql(f"""
    UPDATE workspace.default.fact_orders_update_demo
    SET amount = amount + 100
    WHERE order_id = {demo_id}
""")

print("Demo order updated!")

Demo order updated!


In [0]:
display(
    spark.table("workspace.default.fact_orders_update_demo")
    .filter(col("order_id") == demo_id)
)

order_id,date_key,customer_key,product_key,payment_key,status_key,quantity,amount,source
100001,20260901,268,3,1,2,1,77735.24,web


## Notebook 04 – Delta Tables

## Objectives
The objective of this notebook was to create Delta tables, perform incremental data loading, and demonstrate Delta Lake time travel.

## Work Completed
1. Loaded six warehouse CSV files into Databricks using PySpark.
2. Created six managed Delta tables in the `workspace.default` schema.
3. Examined Delta table history and version information.
4. Created a demonstration table and tested an update.
5. Loaded 8,593 incremental PostgreSQL order records.
6. Validated the incremental data and confirmed that all dimension keys were present.
7. Used Delta MERGE to update 8,593 existing orders.
8. Used time travel to compare table versions before and after incremental loading.

## Result
The MERGE operation successfully updated 8,593 existing order records without inserting duplicate orders. Delta Lake history recorded the operation as version 2.

## Conclusion
This notebook demonstrated how Delta Lake supports reliable incremental data loading, transaction history, and time travel for an online shop data warehouse.